# 4.1 Branch History and Hardware Interface

This notebook translates the ARTERY predictor idea into tutorial-level code and connects it to the hardware interface files. It avoids the unfinished TODO cells from the original notebook and presents a compact BHT-style predictor.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

DATA_PATHS = [
    Path('./s21_data.mat'),
    Path('../s21_data.mat'),
    Path('../../software/s21_data.mat'),
]

def find_s21_data():
    for p in DATA_PATHS:
        if p.exists():
            return p
    raise FileNotFoundError('Put s21_data.mat in the notebook directory or artery/software/.')

def load_s21():
    import scipy.io as sio
    data_path = find_s21_data()
    read_data = sio.loadmat(data_path)
    read_zero = read_data['data'][0]
    read_one = read_data['data'][1]
    read_zero_i, read_zero_q = read_zero[:, :, 0], read_zero[:, :, 1]
    read_one_i, read_one_q = read_one[:, :, 0], read_one[:, :, 1]
    return read_data, read_zero_i, read_zero_q, read_one_i, read_one_q

def demod_part(omega, read_i, read_q, phase=0.0):
    assert read_i.shape == read_q.shape
    ts = np.arange(read_i.shape[1])
    cos_ = np.cos(omega * ts + phase)[None, :]
    sin_ = np.sin(omega * ts + phase)[None, :]
    sum_i = np.sum(read_i * cos_ + read_q * sin_, axis=1)
    sum_q = np.sum(read_q * cos_ - read_i * sin_, axis=1)
    return np.column_stack([sum_i, sum_q])

OMEGAS = 2 * np.pi * (np.array([6.881, 6.79525, 6.97284]) - 7)

In [ ]:
read_data, read_zero_i, read_zero_q, read_one_i, read_one_q = load_s21()
window_start = 850
step = 100
num_steps = 16
omega = OMEGAS[0]

def trajectory_bits(read_i, read_q, center_zero, center_one, shots):
    bits = []
    probs = []
    for k in range(num_steps):
        length = (k + 1) * step
        feat = demod_part(omega, read_i[:shots, window_start:window_start + length], read_q[:shots, window_start:window_start + length])
        d0 = np.linalg.norm(feat - center_zero, axis=1)
        d1 = np.linalg.norm(feat - center_one, axis=1)
        p1 = d0 / (d0 + d1 + 1e-12)
        bits.append((p1 >= 0.5).astype(np.uint8))
        probs.append(p1)
    return np.stack(bits, axis=1), np.stack(probs, axis=1)

train_shots = 1000
full_len = num_steps * step
train_zero = demod_part(omega, read_zero_i[:train_shots, window_start:window_start + full_len], read_zero_q[:train_shots, window_start:window_start + full_len])
train_one = demod_part(omega, read_one_i[:train_shots, window_start:window_start + full_len], read_one_q[:train_shots, window_start:window_start + full_len])
center_zero, center_one = train_zero.mean(axis=0), train_one.mean(axis=0)

bits0, probs0 = trajectory_bits(read_zero_i, read_zero_q, center_zero, center_one, train_shots)
bits1, probs1 = trajectory_bits(read_one_i, read_one_q, center_zero, center_one, train_shots)
train_bits = np.vstack([bits0, bits1])
train_labels = np.array([0] * train_shots + [1] * train_shots)

history_len = 8
bht = {}
for bits, label in zip(train_bits, train_labels):
    for end in range(history_len, bits.shape[0] + 1):
        key = ''.join(map(str, bits[end - history_len:end]))
        zeros, ones = bht.get(key, [0, 0])
        if label == 0:
            zeros += 1
        else:
            ones += 1
        bht[key] = [zeros, ones]

bht_prob = {key: ones / (zeros + ones) for key, (zeros, ones) in bht.items()}
print('BHT entries:', len(bht_prob))
print('example entries:', list(bht_prob.items())[:5])

In [ ]:
test_shots = 500
test_bits0, _ = trajectory_bits(read_zero_i[train_shots:train_shots + test_shots], read_zero_q[train_shots:train_shots + test_shots], center_zero, center_one, test_shots)
test_bits1, _ = trajectory_bits(read_one_i[train_shots:train_shots + test_shots], read_one_q[train_shots:train_shots + test_shots], center_zero, center_one, test_shots)
test_bits = np.vstack([test_bits0, test_bits1])
test_labels = np.array([0] * test_shots + [1] * test_shots)

threshold_hi = 0.9
threshold_lo = 0.1
pred_labels = []
decision_steps = []
for bits in test_bits:
    pred = None
    step_idx = num_steps
    for end in range(history_len, bits.shape[0] + 1):
        key = ''.join(map(str, bits[end - history_len:end]))
        p1 = bht_prob.get(key, 0.5)
        if p1 >= threshold_hi:
            pred, step_idx = 1, end
            break
        if p1 <= threshold_lo:
            pred, step_idx = 0, end
            break
    if pred is None:
        pred = int(bits[-1])
    pred_labels.append(pred)
    decision_steps.append(step_idx)

pred_labels = np.array(pred_labels)
print('BHT-style prediction accuracy:', float(np.mean(pred_labels == test_labels)))
print('mean decision samples:', float(np.mean(decision_steps) * step))
print('thresholds:', threshold_lo, threshold_hi)

## Interface Files

The full Vivado project is intentionally not included. The tutorial repository keeps only the interface-level contract needed to explain and reproduce the data boundary.

In [ ]:
from pathlib import Path
for path in sorted(Path('../../hw/interface').glob('*')):
    print(path)

print('
--- feedback_datapath.md ---')
print(Path('../../hw/interface/feedback_datapath.md').read_text()[:1500])

## GUI Result Figure

![GUI result](../results/artery_gui_current.png)

## Hardware Packet Meaning

The board-side result should include the selected branch, decision confidence, decision sample index, latency counter, and feedback waveform identifier or waveform samples. This lets the host verify both the prediction and the feedback waveform selected by the ARTERY-style logic.